# TurboQuant for LLM KV Cache Compression

Google's TurboQuant paper heavily targets **KV Cache compression** in Large Language Models. 

In LLM generation, storing the Keys (K) and Values (V) for past tokens takes up massive amounts of GPU memory. By compressing K and V from 16-bit floats to 4-bit (or fewer) integers, we can drastically increase the context window and batch size.

This notebook demonstrates how to simulate KV Cache compression using `tqtorch` and measures the memory savings and attention fidelity.

In [29]:
import torch
import torch.nn.functional as F
from tqtorch import MSEQuantizer, InnerProductQuantizer

In [30]:
# 1. Simulate an LLM Attention Layer's KV Cache
batch_size = 1
num_heads = 32
seq_len = 4096       # A moderately long context
head_dim = 128

# Simulate intermediate representations (usually float16 or bfloat16)
dtype = torch.float32  # using float32 for stable random init, but logic applies to fp16/bf16
Q = torch.randn(batch_size, num_heads, 1, head_dim, dtype=dtype)        # Query for the current token
K_cache = torch.randn(batch_size, num_heads, seq_len, head_dim, dtype=dtype)  # Past Keys
V_cache = torch.randn(batch_size, num_heads, seq_len, head_dim, dtype=dtype)  # Past Values

print(f"Uncompressed KV Cache Size: {(K_cache.numel() + V_cache.numel()) * 4 / (1024**2):.2f} MB (if FP32)")
print(f"Uncompressed KV Cache Size: {(K_cache.numel() + V_cache.numel()) * 2 / (1024**2):.2f} MB (if FP16/BF16)")

Uncompressed KV Cache Size: 128.00 MB (if FP32)
Uncompressed KV Cache Size: 64.00 MB (if FP16/BF16)


In [31]:
# 2. Calculate Exact Attention (Baseline)
# Attention Scores: Softmax(Q * K^T / sqrt(d))
exact_scores = torch.matmul(Q, K_cache.transpose(-2, -1)) / (head_dim ** 0.5)
exact_probs = F.softmax(exact_scores, dim=-1)

# Attention Output: Probs * V
exact_output = torch.matmul(exact_probs, V_cache)

print("Computed exact baseline attention.")

Computed exact baseline attention.


In [36]:
# 3. Compress the KV Cache with TurboQuant
bits = 4

# Keys are used in dot products (Q * K^T), so we use the InnerProductQuantizer (Algorithm 2)
k_quantizer = InnerProductQuantizer(dim=head_dim, bits=bits)

# Values are just retrieved/averaged, so we only need MSE preservation (Algorithm 1)
v_quantizer = MSEQuantizer(dim=head_dim, bits=bits)

# Reshape caches to 2D for quantization, then back to original shape concept
K_flat = K_cache.view(-1, head_dim)
V_flat = V_cache.view(-1, head_dim)

K_compressed = k_quantizer.quantize(K_flat)
V_compressed = v_quantizer.quantize(V_flat)

# Memory footprint after compression
k_bytes = K_flat.shape[0] * k_quantizer.bytes_per_vector()
v_bytes = V_flat.shape[0] * v_quantizer.bytes_per_vector()
total_compressed_mb = (k_bytes + v_bytes) / (1024**2)

print(f"Compressed KV Cache Size: {total_compressed_mb:.2f} MB")
print(f"Compression Ratio (vs FP16): {(K_cache.numel() * 2 / k_bytes):.1f}x")

Compressed KV Cache Size: 16.75 MB
Compression Ratio (vs FP16): 3.8x


In [37]:
# 4. Calculate Approximate Attention using Compressed Cache
# Q * K^T using the Inner Product Estimator
Q_flat = Q.view(-1, head_dim)  # (32, 128)
 
# We need the full cross-product matrix: (32 queries x 131072 keys).
approx_scores_cross = k_quantizer.estimate_inner_products(K_compressed, Q_flat)

# Shape is (32, 131072). We reshape to (num_query_heads, num_key_heads, seq_len)
approx_scores_full = approx_scores_cross.view(num_heads, num_heads, seq_len)

# We only want matching heads: Q_head_i @ K_head_i
approx_scores_matched = approx_scores_full[torch.arange(num_heads), torch.arange(num_heads), :]

# Reshape back to attention format and apply scale
approx_scores = approx_scores_matched.view(batch_size, num_heads, 1, seq_len) / (head_dim ** 0.5)

approx_probs = F.softmax(approx_scores, dim=-1)

# Dequantize V for the final multiply
V_approx_flat = v_quantizer.dequantize(V_compressed)
V_approx = V_approx_flat.view(batch_size, num_heads, seq_len, head_dim)

approx_output = torch.matmul(approx_probs, V_approx)

print("Computed approximate attention from compressed cache.")

Computed approximate attention from compressed cache.


In [38]:
# 5. Check Fidelity / Quality Loss
cos_sim = F.cosine_similarity(exact_output.view(-1), approx_output.view(-1), dim=0)
mse = F.mse_loss(exact_output, approx_output)

print("--- Fidelity Results ---")
print(f"Cosine Similarity of final Attention Output: {cos_sim.item():.4f}")
print(f"Mean Squared Error: {mse.item():.6f}")

if cos_sim > 0.95:
    print("\nSuccess! The 8x compressed cache provides NEARLY IDENTICAL attention outputs.")

--- Fidelity Results ---
Cosine Similarity of final Attention Output: 0.9703
Mean Squared Error: 0.000040

Success! The 8x compressed cache provides NEARLY IDENTICAL attention outputs.
